In [1]:
import numpy as np 
import pandas as pd 
import plotly.express as px 
import plotly.graph_objects as go 
import plotly.offline as pyo
import plotly.figure_factory as ff
import dash 
import dash_core_components as dcc
import dash_html_components as html
from dash.dependencies import Input, Output


/var/folders/_m/qzwv1wg53p96cq3kbx83vxbw0000gn/T/ipykernel_82333/83360461.py:8: UserWarning:


The dash_core_components package is deprecated. Please replace
`import dash_core_components as dcc` with `from dash import dcc`

/var/folders/_m/qzwv1wg53p96cq3kbx83vxbw0000gn/T/ipykernel_82333/83360461.py:9: UserWarning:


The dash_html_components package is deprecated. Please replace
`import dash_html_components as html` with `from dash import html`



In [2]:
call = dash.Dash()

call.layout = html.Div([
    dcc.Input(id = 'f1', value = 'hello world', type = 'text'),
    html.Div(id = 'div_1')
]
)
@call.callback(
    Output(component_id= 'div_1', component_property= 'children'),
    [Input(component_id= 'f1', component_property= 'value')],
    prevent_initial_call=True
)

def update_output(input_v):
    return f'you enterned {input_v}'

# call.run()

In [3]:
c1_data = pd.read_csv('../data/gapminderDataFiveYear.csv')
c1_data.head()


,country,year,pop,continent,lifeExp,gdpPercap
0,Afghanistan,1952,8425333.0,Asia,28.801,779.445314
1,Afghanistan,1957,9240934.0,Asia,30.332,820.853030
2,Afghanistan,1962,10267083.0,Asia,31.997,853.100710
3,Afghanistan,1967,11537966.0,Asia,34.020,836.197138
4,Afghanistan,1972,13079460.0,Asia,36.088,739.981106


In [4]:
year_d = [i for i in c1_data.year.drop_duplicates()]

year_d

[1952, 1957, 1962, 1967, 1972, 1977, 1982, 1987, 1992, 1997, 2002, 2007]

In [5]:
color_map = dict(Asia = 'red',
                 Europe = 'orange',
                 Africa = 'green',
                 Americas = 'blue',
                 Oceania = 'pink')

c1 = dash.Dash()

c1.layout = html.Div([
    dcc.Graph(
        id = 'call_1'
    ),
    dcc.Dropdown(id = 'drop_year', 
                 options  = [dict(label = i, value = i) for i in year_d],
                 value = 1952
                 )
])

@c1.callback(Output(component_id= 'call_1', component_property= 'figure'),
             [Input(component_id= 'drop_year', component_property= 'value')])

def plot_out(select_year):
    df_t = c1_data.loc[c1_data.year == select_year,]
    plot_data = [go.Scatter(x = df_t.loc[df_t.continent == ct,'gdpPercap'],
                            y =  df_t.loc[df_t.continent == ct,'lifeExp'],
                            mode = 'markers',
                            name = ct,
                            text = df_t.country,
                            marker = dict(color = color_map[ct])
                            ) for ct in df_t.continent.drop_duplicates()]
    plot_layout = go.Layout( title = 'happy',
                            xaxis= dict(title = 'GDP per capita'),
                            yaxis= dict(title = 'Life long'),
                            hovermode= 'closest'
                            
    )
    return dict(data = plot_data,
                layout = plot_layout)

# c1.run()

In [6]:
c2_data = pd.read_csv('../data/mpg.csv', header = 0)

c2_data.head()

,mpg,cylinders,displacement,horsepower,weight,acceleration,model_year,origin,name
0,18.0,8,307.0,130,3504,12.0,70,1,chevrolet chevelle malibu
1,15.0,8,350.0,165,3693,11.5,70,1,buick skylark 320
2,18.0,8,318.0,150,3436,11.0,70,1,plymouth satellite
3,16.0,8,304.0,150,3433,12.0,70,1,amc rebel sst
4,17.0,8,302.0,140,3449,10.5,70,1,ford torino


In [7]:
c2 = dash.Dash()

c2.layout = html.Div(id = 'lay1', 
                     children=[
                         html.Div(id = 'select_fliters',
                                  children = [
                                      html.Div(id = 'x_filter',
                                               children= [
                                                   html.Title('X'),
                                                   dcc.Dropdown(id = 'x_name', 
                                                    options= [dict(label = i,value = i) for i in c2_data.columns[:-1]],
                                                    value = 'mpg' )],
                                                style= dict(width = '48%', 
                                                            display = 'inline-block')
                                                   ),
                                        html.Div(id = 'y_filter',
                                                children= [
                                                    html.Title('Y'),
                                                    dcc.Dropdown(id = 'y_name', 
                                                        options= []
                                                    )],
                                                style= dict(width = '48%', 
                                                            float = 'right',
                                                            display = 'inline-block')
                                                    )]
             ),
                        dcc.Graph( id = 'hhh', figure=[])
]
)
@c2.callback(Output(component_id= 'y_name', component_property= 'options'),
             [Input(component_id= 'x_name', component_property= 'value')],
             prevent_initial_call = True)

def new_y(vle):
    tp1 = []
    for i in c2_data.columns[:-1]:
        if i != vle:
            tp1.append(dict(label = i,value = i))
    return tp1

@c2.callback(Output(component_id= 'hhh', component_property= 'figure'),
             [Input(component_id= 'x_name', component_property= 'value'),
              Input(component_id= 'y_name', component_property= 'value')])

def plot_hhh(xn, yn):
    print(type(yn))
    if yn is not None :
        c_data2 = [go.Scatter(x = c2_data[xn],
                            y = c2_data[yn],
                            text = c2_data.name,
                            mode = 'markers',
                            marker= dict(color = 'blue',
                                         opacity = 0.5))]
        c_lay = go.Layout(title = 'happy plot',
                        xaxis= dict(title = xn),
                        yaxis= dict(title = yn),
                        hovermode= 'closest')
        return dict(data = c_data2, layout = c_lay)
    


# c2.run()

In [8]:
[i for i in c2_data.columns[:-1]]

['mpg',
 'cylinders',
 'displacement',
 'horsepower',
 'weight',
 'acceleration',
 'model_year',
 'origin']

In [9]:
[{i:i} for i in c2_data.columns]

[{'mpg': 'mpg'},
 {'cylinders': 'cylinders'},
 {'displacement': 'displacement'},
 {'horsepower': 'horsepower'},
 {'weight': 'weight'},
 {'acceleration': 'acceleration'},
 {'model_year': 'model_year'},
 {'origin': 'origin'},
 {'name': 'name'}]